# 04 — NUTS with Hessian-inverse mass matrix

**Hypothesis H2**: Use the regularized Hessian inverse as the initial mass matrix,
coherent with Stan's M = covariance convention.

**Construction**:
$$M = V \operatorname{diag}\!\left(\frac{1}{\max(0,\lambda_i) + 1/\sigma^2}\right) V^\top$$

- Non-degenerate directions (large $\lambda_i$): eigenvalue ≈ $1/\lambda_i$ (tight, informed by curvature)
- Degenerate directions ($\lambda_i \approx 0$): eigenvalue ≈ $\sigma^2 = 100$ (prior width, uninformed)

**Prediction**: Better than identity start (gives adaptation a head start in
non-degenerate directions), but still won't fix ACF in degenerate directions
because singular geometry has non-constant curvature that no single matrix captures.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Resolve project root
cwd = Path.cwd().resolve()
project_root = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "packages" / "pytorch_models" / "markov_transformer.py").exists()
    ),
    None,
)
if project_root is None:
    alt_root = cwd / "projects" / "markov-chain-learning"
    if (alt_root / "packages" / "pytorch_models" / "markov_transformer.py").exists():
        project_root = alt_root

if project_root is None:
    raise RuntimeError("Could not locate markov-chain-learning project root")

packages_dir = project_root / "packages"
if str(packages_dir) not in sys.path:
    sys.path.insert(0, str(packages_dir))

from pytorch_models import MarkovTransformer

DATA_DIR = project_root / "experiments" / "single-chain" / "data"
print(f"Project root: {project_root}")
print(f"Data dir: {DATA_DIR}")

## Load data & model

In [ ]:
# Load dataset
data = torch.load(DATA_DIR / "sequences.pt", weights_only=False)
sequences = data["sequences"]
data_cfg = data["config"]

VOCAB_SIZE = int(data_cfg["n_states"])
MAX_LEN = int(data_cfg["L"])
PAD_ID = int(data_cfg.get("pad_id", -1))
DGP_REGIME = data_cfg.get("dgp_regime", "unknown")

# Load trained model
device = torch.device("cpu")
D_MODEL = VOCAB_SIZE * 2

model = MarkovTransformer(vocab_size=VOCAB_SIZE, d_model=D_MODEL, max_len=MAX_LEN).to(
    device
)
ckpt = torch.load(DATA_DIR / "checkpoint_single_chain.pt", weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()

# Prepare data
x_data = sequences[:, :-1].to(device).clone()
y_data = sequences[:, 1:].to(device)
x_data[x_data == PAD_ID] = 0

mle_param = torch.cat([p.flatten() for p in model.parameters()]).detach()
d = mle_param.shape[0]

print(f"DGP regime: {DGP_REGIME}")
print(f"Model: {d:,} parameters, checkpoint epoch {ckpt['epoch']}")

In [ ]:
SIGMA_PRIOR = 10.0


def loss_fn(logits, targets):
    ce = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        targets.reshape(-1),
        reduction="none",
        ignore_index=PAD_ID,
    )
    ce = ce.view(logits.shape[:-1])
    mask = (targets != PAD_ID).float()
    return (ce * mask).sum() / mask.sum()


def make_prior_logp(mu: torch.Tensor, sigma=10.0):
    mean = mu.detach().clone()

    def prior_logp(params):
        flat = torch.cat([p.flatten() for p in params])
        diff = flat - mean
        return -0.5 * diff.pow(2).sum() / (sigma**2)

    return prior_logp

## Hessian computation & eigenspectrum visualization

The key question: what does the loss landscape look like at the MLE?
We visualize the Hessian eigenspectrum at each stage of the mass matrix construction.

In [ ]:
from torch_bdn.bn.bayesian_net import approx_hessian

print("Computing Hessian at MLE...")
H = approx_hessian(model, loss_fn, x_data, y_data, chunk_size=10240).cpu()
eigvals, eigvecs = torch.linalg.eigh(H)
print(f"Done. Eigenvalues: [{eigvals[0]:.4e}, {eigvals[-1]:.4e}]")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EIGENVALUE SPECTRUM — the core diagnostic
# ══════════════════════════════════════════════════════════════════════════════
# Sort descending for visualization
idx_desc = torch.argsort(eigvals, descending=True)
lam_sorted = eigvals[idx_desc]

prior_precision = 1.0 / SIGMA_PRIOR**2

# Three stages:
# 1. Raw Hessian eigenvalues (may have negatives from numerical error)
# 2. Clipped + regularized: max(0, λ) + 1/σ²  (= precision eigenvalues)
# 3. Inverse: 1 / (max(0, λ) + 1/σ²)          (= covariance eigenvalues = mass matrix)
lam_raw = lam_sorted.numpy()
lam_clipped_reg = (lam_sorted.clamp(min=0) + prior_precision).numpy()
lam_inv = (1.0 / (lam_sorted.clamp(min=0) + prior_precision)).numpy()

# Identify degenerate boundary
eigval_threshold = eigvals.max().item() * 1e-3
n_nd = int((lam_sorted.abs() > eigval_threshold).sum())
n_degen = d - n_nd

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Raw Hessian eigenvalues
ax = axes[0]
ax.semilogy(np.abs(lam_raw), color="steelblue", lw=0.8, label="|λ| (raw Hessian)")
neg_mask = lam_raw < 0
if neg_mask.any():
    ax.scatter(
        np.where(neg_mask)[0],
        np.abs(lam_raw[neg_mask]),
        color="red",
        s=8,
        zorder=5,
        label=f"negative ({neg_mask.sum()})",
    )
ax.axvline(
    n_nd - 0.5,
    color="lime",
    lw=1.5,
    ls="--",
    alpha=0.7,
    label=f"degen boundary ({n_nd} ND / {n_degen} DG)",
)
ax.axhline(
    prior_precision, color="orange", lw=1, ls=":", label=f"1/σ² = {prior_precision:.0e}"
)
ax.set_ylabel("Eigenvalue (log scale)")
ax.set_title("Stage 1: Raw Hessian eigenvalues at MLE", fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: Clipped + regularized (precision)
ax = axes[1]
ax.semilogy(lam_clipped_reg, color="darkorange", lw=0.8, label="max(0, λ) + 1/σ²")
ax.axvline(n_nd - 0.5, color="lime", lw=1.5, ls="--", alpha=0.7)
ax.axhline(
    prior_precision,
    color="orange",
    lw=1,
    ls=":",
    label=f"floor = 1/σ² = {prior_precision:.0e}",
)
cond_prec = lam_clipped_reg[0] / lam_clipped_reg[-1]
ax.set_ylabel("Eigenvalue (log scale)")
ax.set_title(
    f"Stage 2: Clipped + prior regularized (PRECISION)  —  cond = {cond_prec:.0f}",
    fontweight="bold",
)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 3: Inverse (covariance = mass matrix)
ax = axes[2]
ax.semilogy(lam_inv, color="darkgreen", lw=0.8, label="1 / (max(0, λ) + 1/σ²)")
ax.axvline(n_nd - 0.5, color="lime", lw=1.5, ls="--", alpha=0.7)
ax.axhline(
    SIGMA_PRIOR**2,
    color="purple",
    lw=1,
    ls=":",
    label=f"ceiling = σ² = {SIGMA_PRIOR**2:.0f}",
)
cond_cov = lam_inv.max() / lam_inv.min()
ax.set_ylabel("Eigenvalue (log scale)")
ax.set_xlabel("Direction index (sorted by Hessian eigenvalue, largest first)")
ax.set_title(
    f"Stage 3: Inverse = COVARIANCE (mass matrix M)  —  cond = {cond_cov:.0f}",
    fontweight="bold",
)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)

fig.suptitle(
    f"Mass matrix construction pipeline  (d={d}, σ={SIGMA_PRIOR})\n"
    f"{n_nd} non-degenerate + {n_degen} degenerate directions",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

print("\nSummary:")
print(
    f"  Raw Hessian: λ ∈ [{lam_raw[-1]:.2e}, {lam_raw[0]:.2e}], {neg_mask.sum()} negative"
)
print(
    f"  Precision (clipped+reg): [{lam_clipped_reg[-1]:.2e}, {lam_clipped_reg[0]:.2e}], cond={cond_prec:.0f}"
)
print(
    f"  Covariance (inverse):    [{lam_inv.min():.2e}, {lam_inv.max():.2e}], cond={cond_cov:.0f}"
)
print(f"  Degenerate dirs get M eigenvalue ≈ σ² = {SIGMA_PRIOR**2} (prior width)")
print("  Non-degen dirs get M eigenvalue ≈ 1/λ_H (posterior width)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ZOOM: what does the transition region look like?
# ══════════════════════════════════════════════════════════════════════════════
# The interesting region is around the degenerate boundary where eigenvalues
# transition from "informed by curvature" to "floored at prior precision"

window = 50  # show 50 directions on each side of boundary
i_start = max(0, n_nd - window)
i_end = min(d, n_nd + window)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: precision eigenvalues near boundary
ax = axes[0]
x_range = range(i_start, i_end)
ax.semilogy(x_range, lam_clipped_reg[i_start:i_end], "o-", ms=3, color="darkorange")
ax.axvline(n_nd - 0.5, color="lime", lw=2, ls="--", label="degen boundary")
ax.axhline(
    prior_precision,
    color="orange",
    lw=1.5,
    ls=":",
    label=f"1/σ² = {prior_precision:.0e}",
)
ax.set_xlabel("Direction index")
ax.set_ylabel("Precision eigenvalue")
ax.set_title("Precision near boundary")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: covariance eigenvalues near boundary
ax = axes[1]
ax.semilogy(x_range, lam_inv[i_start:i_end], "o-", ms=3, color="darkgreen")
ax.axvline(n_nd - 0.5, color="lime", lw=2, ls="--", label="degen boundary")
ax.axhline(
    SIGMA_PRIOR**2, color="purple", lw=1.5, ls=":", label=f"σ² = {SIGMA_PRIOR**2}"
)
ax.set_xlabel("Direction index")
ax.set_ylabel("Covariance eigenvalue (= M eigenvalue)")
ax.set_title("Mass matrix eigenvalues near boundary")
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle(
    f"Zoom: ±{window} directions around degenerate boundary (index {n_nd})",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPARISON: if we have the empirically adapted M from the no-hessian run,
# compare its eigenspectrum to our analytical construction
# ══════════════════════════════════════════════════════════════════════════════
empirical_M_path = DATA_DIR / "adapted_mass_matrix_no_hessian.pt"

if empirical_M_path.exists():
    M_empirical = torch.load(empirical_M_path, weights_only=True)
    eig_empirical = (
        torch.linalg.eigvalsh(M_empirical).flip(0).numpy()
    )  # sorted descending

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.semilogy(
        lam_inv,
        color="darkgreen",
        lw=1.2,
        label="Analytical: 1/(max(0,λ)+1/σ²)",
        alpha=0.8,
    )
    ax.semilogy(
        eig_empirical,
        color="crimson",
        lw=1.2,
        label="Empirical (from no-hessian warmup)",
        alpha=0.8,
    )
    ax.axvline(
        n_nd - 0.5, color="lime", lw=1.5, ls="--", alpha=0.7, label="degen boundary"
    )
    ax.set_xlabel("Direction index (sorted by magnitude, largest first)")
    ax.set_ylabel("Mass matrix eigenvalue (log)")
    ax.set_title(
        "Analytical vs Empirical mass matrix eigenspectra",
        fontweight="bold",
        fontsize=13,
    )
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Eigenvector alignment: how well do the two bases agree?
    # Compute |V_hessian^T @ V_empirical|^2 — if aligned, this is ~identity
    eig_emp_vals, eig_emp_vecs = torch.linalg.eigh(M_empirical)
    # Sort empirical by descending eigenvalue
    emp_desc = torch.argsort(eig_emp_vals, descending=True)
    V_emp = eig_emp_vecs[:, emp_desc]
    V_hess = eigvecs[:, idx_desc]

    # Top-k alignment
    k = min(50, n_nd)
    overlap = (V_hess[:, :k].T @ V_emp[:, :k]).abs()  # (k, k)
    print(f"\nEigenvector alignment (top {k} directions):")
    print(f"  Mean |<v_hess, v_emp>|: {overlap.diagonal().mean():.3f}")
    print(
        f"  Max off-diagonal:       {(overlap - torch.diag(overlap.diagonal())).max():.3f}"
    )
else:
    print(f"No empirical mass matrix found at {empirical_M_path.name}")
    print("Run the no-hessian notebook first to generate it.")

## Construct and save the analytical mass matrix

In [ ]:
# Build M_posterior_inv = V diag(1 / (max(0,λ) + 1/σ²)) Vᵀ
eigvals_clamped = eigvals.clamp(min=0.0)
m_eig = eigvals_clamped + prior_precision
m_inv_eig = 1.0 / m_eig
M_hessian_inv = eigvecs @ torch.diag(m_inv_eig) @ eigvecs.T

# Verify PSD
eig_check = torch.linalg.eigvalsh(M_hessian_inv)
assert (eig_check > 0).all(), "Mass matrix is not PSD!"

cond = eig_check.max() / eig_check.min()
print(f"✓ Analytical mass matrix constructed (d={d})")
print(f"  Eigenvalues: [{eig_check.min():.4e}, {eig_check.max():.4e}]")
print(f"  Condition number: {cond:.1f}")
print("  Non-degen dirs: eigenvalue ≈ 1/λ_H (tight)")
print(f"  Degenerate dirs: eigenvalue = σ² = {SIGMA_PRIOR**2} (prior width)")

## NUTS — iterative warmup with Hessian-inverse initialization

Same workflow as the no-hessian notebook:
- `START_FRESH=True` → use analytical M_hessian_inv
- `START_FRESH=False` → load last adapted M from disk
- Re-run warmup cell as needed

In [ ]:
import collections

from torch_bdn.bn import BayesianNet
from torch_bdn.sampling import NUTS, Perturb, Sampler

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
START_FRESH = True  # True → analytical M_hessian_inv; False → load adapted from disk

N_WARMUP = 5000
MAX_TREE_DEPTH = 7
MASS_MATRIX_PATH = DATA_DIR / "adapted_mass_matrix_hessian_inv.pt"

# Load or initialize mass matrix
if START_FRESH:
    mass_matrix = M_hessian_inv
    print(f"Starting from ANALYTICAL Hessian-inverse mass matrix (d={d})")
    print(f"  cond={cond:.1f}")
else:
    if MASS_MATRIX_PATH.exists():
        mass_matrix = torch.load(MASS_MATRIX_PATH, weights_only=True)
        svd = torch.linalg.svdvals(mass_matrix)
        print(f"Loaded adapted mass matrix from {MASS_MATRIX_PATH.name}")
        print(f"  shape={tuple(mass_matrix.shape)}, cond={svd[0] / svd[-1]:.1f}")
    else:
        mass_matrix = M_hessian_inv
        print("⚠ No saved matrix found, falling back to analytical M_hessian_inv")

bn = BayesianNet(
    model, loss_fn, make_prior_logp(mle_param, sigma=SIGMA_PRIOR), compile=True
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WARMUP ONLY — re-run this cell as many times as you want
# ══════════════════════════════════════════════════════════════════════════════

warmup_result = Sampler(bn, x_data, y_data).sample(
    config=NUTS(
        n_warmup=N_WARMUP,
        step_size=0.01,
        max_tree_depth=MAX_TREE_DEPTH,
        target_accept=0.69,
        mass_matrix=mass_matrix,
        adapt_mass_matrix=True,
    ),
    n_samples=1,
    n_chains=1,
    init_strategy=Perturb(scale=0.1),
    n_cores=1,
)

# Extract adapted mass matrix
diag = warmup_result.chains[0].diagnostics
adapted_M = diag.get("adapted_mass_matrix")
adapted_step_size = diag.get("adapted_step_size", diag["step_size"])

if adapted_M is not None:
    mass_matrix = adapted_M
    torch.save(adapted_M, MASS_MATRIX_PATH)
    svd = torch.linalg.svdvals(adapted_M)
    print(f"✓ Warmup complete. Saved to {MASS_MATRIX_PATH.name}")
    print(f"  cond={svd[0] / svd[-1]:.1f}, σ_min={svd[-1]:.4e}, σ_max={svd[0]:.4e}")
    print(f"  adapted ε = {adapted_step_size:.4e}")
    print(f"  accept = {warmup_result.chains[0].acceptance_rate:.3f}")
else:
    print("⚠ No adapted mass matrix returned")
    print(
        f"  ε = {adapted_step_size:.4e}, accept = {warmup_result.chains[0].acceptance_rate:.3f}"
    )

for mu in diag.get("warmup_mass_updates", []):
    print(
        f"    M update @ step {mu['step']}: type={mu['type']}, n_draws={mu['n_draws']}"
    )

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PRODUCTION SAMPLING
# ══════════════════════════════════════════════════════════════════════════════
N_SAMPLES = 2000

mass_matrix_prod = torch.load(MASS_MATRIX_PATH, weights_only=True)
print(f"Loaded adapted mass matrix from {MASS_MATRIX_PATH.name}")

result = Sampler(bn, x_data, y_data).sample(
    config=NUTS(
        n_warmup=0,
        step_size=adapted_step_size,
        max_tree_depth=MAX_TREE_DEPTH,
        target_accept=0.69,
        mass_matrix=mass_matrix_prod,
        adapt_mass_matrix=False,
        adapt_step_size=False,
    ),
    n_samples=N_SAMPLES,
    n_chains=1,
    init_strategy=Perturb(scale=0.1),
    n_cores=1,
)

# Summary
for ci, ch in enumerate(result.chains):
    diag = ch.diagnostics
    tree_depths = diag.get("tree_depths", [])
    leapfrog_counts = diag.get("leapfrog_counts", [])
    n_hit_max = diag.get("n_hit_max_depth", 0)
    n_div = diag.get("n_divergences", 0)
    n_samp = len(tree_depths)
    print(
        f"Chain {ci}: accept={ch.acceptance_rate:.3f}  "
        f"ε={diag.get('step_size', '?'):.4e}  "
        f"mean_depth={diag.get('mean_tree_depth', 0):.1f}  "
        f"mean_L={diag.get('mean_leapfrog', 0):.0f}"
    )
    print(f"  max_depth_hits={n_hit_max}/{n_samp}  divergences={n_div}/{n_samp}")
    if tree_depths:
        depth_counts = collections.Counter(tree_depths)
        print(f"  Tree depth distribution: {dict(sorted(depth_counts.items()))}")

In [ ]:
# ── Persist production samples ──
nuts_path = DATA_DIR / "nuts_samples_hessian_inv.pt"

chains_payload = [
    {
        "parameters": torch.stack(ch.parameters).cpu(),
        "acceptance_rate": ch.acceptance_rate,
        "diagnostics": ch.diagnostics,
    }
    for ch in result.chains
]

torch.save(
    {
        "chains": chains_payload,
        "config": {
            "n_chains": 1,
            "n_warmup": N_WARMUP,
            "n_samples": N_SAMPLES,
            "sigma_prior": SIGMA_PRIOR,
            "dgp_regime": DGP_REGIME,
            "mass_matrix_init": "hessian_inv" if START_FRESH else "loaded",
        },
        "mle_param": mle_param.cpu(),
        "eigvals": eigvals.cpu(),
        "eigvecs": eigvecs.cpu(),
    },
    nuts_path,
)
print(f"✓ Saved to {nuts_path}")
print(f"  1 chain × {N_SAMPLES} samples × d={d}")

## ACF analysis

In [ ]:
# ── ACF per Hessian eigendirection ──
V = eigvecs[:, idx_desc].cpu().float()
MAX_LAG = min(1500, N_SAMPLES // 4)


def compute_acf_eigenbasis(samples_tensor, V, max_lag):
    """Project samples into eigenbasis, compute ACF per direction via FFT."""
    z = samples_tensor @ V
    d = z.shape[1]
    acf = torch.zeros(d, max_lag)
    for j in range(d):
        x = z[:, j]
        x = x - x.mean()
        var = x.var()
        if var < 1e-20:
            continue
        n = x.shape[0]
        padded = torch.zeros(2 * n)
        padded[:n] = x
        ft = torch.fft.rfft(padded)
        acov = torch.fft.irfft(ft * ft.conj())[:n] / n
        acf[j, :max_lag] = acov[:max_lag] / acov[0].clamp(min=1e-20)
    return acf


samps = torch.stack(result.chains[0].parameters).cpu().float()
acf_sorted = compute_acf_eigenbasis(samps, V, MAX_LAG)

fig, ax = plt.subplots(figsize=(16, 8))
im = ax.imshow(
    acf_sorted.numpy(),
    aspect="auto",
    cmap="RdBu_r",
    vmin=-0.3,
    vmax=1.0,
    interpolation="nearest",
    origin="upper",
)
ax.set_xlabel("Lag τ", fontsize=12)
ax.set_ylabel("Eigendirection (sorted by λ, largest at top)", fontsize=12)
ax.set_title(
    f"ACF per Hessian eigendirection — NUTS, Hessian-inv M + adaptation\n"
    f"warmup={N_WARMUP}, {N_SAMPLES} samples, d={d}",
    fontsize=13,
    fontweight="bold",
)

if 0 < n_nd < d:
    ax.axhline(
        n_nd - 0.5, color="lime", lw=2, ls="--", label=f"degen boundary (top {n_nd} ND)"
    )
    ax.legend(loc="upper right", fontsize=10)

plt.colorbar(im, ax=ax, shrink=0.8, label="ACF")
plt.tight_layout()
plt.show()

# Summary statistics
acf_at_10 = acf_sorted[:, min(10, MAX_LAG - 1)]
acf_at_50 = acf_sorted[:, min(50, MAX_LAG - 1)]
acf_at_200 = acf_sorted[:, min(200, MAX_LAG - 1)]
print(f"\nACF at lag 10:   median={acf_at_10.median():.3f},  max={acf_at_10.max():.3f}")
print(f"ACF at lag 50:   median={acf_at_50.median():.3f},  max={acf_at_50.max():.3f}")
print(f"ACF at lag 200:  median={acf_at_200.median():.3f},  max={acf_at_200.max():.3f}")

acf50_nd = acf_sorted[:n_nd, min(50, MAX_LAG - 1)]
acf50_dg = acf_sorted[n_nd:, min(50, MAX_LAG - 1)]
print(
    f"\nACF@50 non-degen ({n_nd} dirs): median={acf50_nd.median():.3f}, max={acf50_nd.max():.3f}"
)
print(
    f"ACF@50 degenerate ({n_degen} dirs): median={acf50_dg.median():.3f}, max={acf50_dg.max():.3f}"
)

In [ ]:
# ── ESS ──
def compute_ess(acf_row, n_samples):
    total = 0.0
    for k in range(acf_row.shape[0]):
        if acf_row[k] < 0:
            break
        total += acf_row[k]
    tau = 1 + 2 * (total - 1)
    return n_samples / max(tau, 1.0)


ess_per_dir = torch.tensor([compute_ess(acf_sorted[i], N_SAMPLES) for i in range(d)])
ess_nd = ess_per_dir[:n_nd]
ess_dg = ess_per_dir[n_nd:]

print(
    f"ESS (all {d} dirs): min={ess_per_dir.min():.1f}, median={ess_per_dir.median():.1f}, max={ess_per_dir.max():.1f}"
)
print(
    f"ESS non-degen ({n_nd}): min={ess_nd.min():.1f}, median={ess_nd.median():.1f}, max={ess_nd.max():.1f}"
)
print(
    f"ESS degenerate ({n_degen}): min={ess_dg.min():.1f}, median={ess_dg.median():.1f}, max={ess_dg.max():.1f}"
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(d), ess_per_dir.numpy(), width=1.0, color="steelblue", alpha=0.7)
if 0 < n_nd < d:
    ax.axvline(n_nd - 0.5, color="lime", lw=2, ls="--", label="degen boundary")
    ax.legend()
ax.set_xlabel("Eigendirection (sorted by λ, largest first)")
ax.set_ylabel("ESS")
ax.set_title(f"Effective Sample Size per eigendirection (N={N_SAMPLES})")
ax.set_xlim(-0.5, d - 0.5)
plt.tight_layout()
plt.show()

## Verdict

- **ACF@50 median < 0.3** → H2 falsified, Hessian-inverse preconditioning works
- **ACF@50 median > 0.5** → H2 confirmed, need non-local methods (PT)
- **Compare non-degen vs degen**: if non-degen improves but degen doesn't, that
  confirms the Hessian helps where curvature is informative but can't help in the
  flat/singular directions